In [ ]:
#part 1

import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv("acoustic features.csv")

print(df.head())
print(df.info())
print(df.shape)
print(df.columns)

X = df.drop(columns=['Class'])
y = df['Class']

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Classes:", set(y_encoded))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled shape:", X_scaled.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=17342
)

   Class  _RMSenergy_Mean  _Lowenergy_Mean  _Fluctuation_Mean  _Tempo_Mean  \
0  relax            0.052            0.591              9.136      130.043   
1  relax            0.125            0.439              6.680      142.240   
2  relax            0.046            0.639             10.578      188.154   
3  relax            0.135            0.603             10.442       65.991   
4  relax            0.066            0.591              9.769       88.890   

   _MFCC_Mean_1  _MFCC_Mean_2  _MFCC_Mean_3  _MFCC_Mean_4  _MFCC_Mean_5  ...  \
0         3.997         0.363         0.887         0.078         0.221  ...   
1         4.058         0.516         0.785         0.397         0.556  ...   
2         2.775         0.903         0.502         0.329         0.287  ...   
3         2.841         1.552         0.612         0.351         0.011  ...   
4         3.217         0.228         0.814         0.096         0.434  ...   

   _Chromagram_Mean_9  _Chromagram_Mean_10  _Chrom

In [16]:
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import RepeatedKFold, cross_val_score
import numpy as np

models = {
    "Linear Classifier": Perceptron(max_iter=1000, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Gaussian NB": GaussianNB(),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)
}

rkf = RepeatedKFold(
    n_splits=10,
    n_repeats=100,
    random_state=42
)


results = {}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_scaled,
        y_encoded,
        cv=rkf,
        scoring="accuracy",
        n_jobs=-1
    )
    
    results[name] = (scores.mean(), scores.std())
    
    print(f"{name:20s} mean={scores.mean():.4f} std={scores.std():.4f}")



Linear Classifier    mean=0.7619 std=0.0695
Logistic Regression  mean=0.7906 std=0.0618
KNN                  mean=0.7152 std=0.0723
Gaussian NB          mean=0.7487 std=0.0718
Neural Network       mean=0.7782 std=0.0657


Part 2

Logistic Regression achieved the highest performance (0.7906 ± 0.0618), indicating that the dataset contains a strong linear structure that can be effectively modeled using a probabilistic linear classifier.  The Neural Network also performed well (0.7782 ± 0.0657), suggesting that some nonlinear relationships exist in the data, but they are not dominant. The Perceptron classifier showed moderate performance but was less stable, likely due to its sensitivity to noise and lack of probabilistic optimization. Gaussian Naive Bayes performed slightly worse, which is expected because the independence assumption between acoustic features is violated in this dataset. K-Nearest Neighbors performed the worst, which is likely due to the high dimensionality of the feature space (50 features). In high dimensions, distance-based methods become less effective, leading to poorer generalization. Overall, linear models performed competitively or better than nonlinear models, suggesting that the dataset is closer to linearly separable than highly complex.

In [18]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedKFold
import numpy as np

rkf = RepeatedKFold(
    n_splits=10,
    n_repeats=10,   # you can reduce from 100 for speed
    random_state=42
)

base_model = LogisticRegression(max_iter=1000, random_state=42)

sfs = SequentialFeatureSelector(
    estimator=base_model,
    n_features_to_select=15,   # you can change this (10–20 is reasonable)
    direction="forward",
    scoring="accuracy",
    cv=rkf,
    n_jobs=-1
)

MLPClassifier(
    hidden_layer_sizes=(64,),
    max_iter=500,
    early_stopping=True,
    random_state=42
)

sfs.fit(X_scaled, y_encoded)

selected_features = np.array(df.drop(columns=["Class"]).columns)[sfs.get_support()]

print("Selected features:")
print(list(selected_features))

X_selected = X_scaled[:, sfs.get_support()]
print(X_selected.shape)

results_fs = {}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_selected,
        y_encoded,
        cv=rkf,
        scoring="accuracy",
        n_jobs=-1
    )
    
    results_fs[name] = (scores.mean(), scores.std())
    
    print(f"{name:20s} mean={scores.mean():.4f} std={scores.std():.4f}")

Selected features:
['_Fluctuation_Mean', '_MFCC_Mean_1', '_Roughness_Slope', '_Zero-crossingrate_Mean', '_AttackTime_Slope', '_Rolloff_Mean', '_Eventdensity_Mean', '_Pulseclarity_Mean', '_Spectralcentroid_Mean', '_EntropyofSpectrum_Mean', '_Chromagram_Mean_3', '_Chromagram_Mean_6', '_HarmonicChangeDetectionFunction_Std', '_HarmonicChangeDetectionFunction_PeriodAmp', '_HarmonicChangeDetectionFunction_PeriodEntropy']
(400, 15)
Linear Classifier    mean=0.7513 std=0.0748
Logistic Regression  mean=0.8315 std=0.0670
KNN                  mean=0.7318 std=0.0657
Gaussian NB          mean=0.7670 std=0.0626


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

Neural Network       mean=0.7833 std=0.0598


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Part 3

Effect of Feature Selection

Feature selection was applied using a forward selection method with Logistic Regression as the base estimator. The goal was to identify a subset of informative features while reducing redundancy in the dataset.

After reducing the feature space from 50 to 15 features, several models showed improved performance. Logistic Regression achieved the highest improvement, increasing from 0.7906 to 0.8315, indicating that removing irrelevant features improved linear separability.

Gaussian Naive Bayes also improved slightly, likely because feature selection reduced violations of the independence assumption. KNN showed modest improvement, but remains limited due to the curse of dimensionality in high-dimensional feature spaces.

The Linear Classifier showed slight degradation, suggesting sensitivity to feature selection choices. Overall, the results indicate that not all features contribute equally to classification performance, and reducing dimensionality can improve both accuracy and stability depending on the algorithm.